[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/google-genai/orchestration-agent.ipynb)


<center>
    <p style="text-align:center">
        <img alt="Phoenix 标志" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">文档</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://join.slack.com/t/arize-ai/shared_invite/zt-3r07iavnk-ammtATWSlF0pSrd1DsMW7g">社区</a>
    </p>
</center>

# Google GenAI SDK - 构建编排智能体

## 安装依赖

In [ ]:
!pip install -q google-genai arize-phoenix-otel openinference-instrumentation-google-genai

## 连接 Arize Phoenix

In [ ]:
import os
from getpass import getpass

from google import genai
from google.genai import types
from phoenix.otel import register

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

tracer_provider = register(auto_instrument=True, project_name="google-genai-orchestrator-agent")
tracer = tracer_provider.get_tracer(__name__)

## 通过 Google Vertex AI 进行身份验证

In [ ]:
!gcloud auth login

In [ ]:
# 使用 Vertex AI API 创建客户端，此处也可以改用 Google GenAI API
client = genai.Client(vertexai=True, project="<填写 GCP 项目 ID>", location="us-central1")

# 编排智能体

首先定义供编排器选择的子智能体，本例中使用工具表示这些子智能体。

In [ ]:
# 为不同专业智能体定义模型
FLASH_MODEL = "gemini-2.0-flash-001"


@tracer.chain()
def call_user_proxy_agent(query, context=""):
    """扮演用户并提供反馈的用户代理智能体。"""
    prompt = f"""你是用户代理助手。请像用户一样对下列内容提供反馈：
    上下文： {context}
    Query: {query}
    从用户视角给出诚实、有建设性的反馈。"""

    response = client.models.generate_content(
        model=FLASH_MODEL,
        contents=prompt,
    )
    return response.text.strip()


@tracer.chain()
def call_flight_planning_agent(query, context=""):
    """帮助查找和推荐航班的航班规划智能体。"""
    prompt = f"""你是航班规划助手。请为以下内容规划航班：
    上下文： {context}
    Query: {query}
    提供详细航班选项，考虑价格、时间与便利性。"""

    response = client.models.generate_content(
        model=FLASH_MODEL,
        contents=prompt,
    )
    return response.text.strip()


@tracer.chain()
def call_hotel_recommendation_agent(query, context=""):
    """推荐住宿的酒店推荐智能体。"""
    prompt = f"""你是酒店推荐助手。请为以下内容推荐住宿：
    上下文： {context}
    Query: {query}
    提供合适的酒店选项，详细说明设施、位置与价格范围。"""

    response = client.models.generate_content(
        model=FLASH_MODEL,
        contents=prompt,
    )
    return response.text.strip()


@tracer.chain()
def call_travel_attraction_agent(query, context=""):
    """推荐游览地点的旅游景点推荐智能体。"""
    prompt = f"""你是旅游景点推荐助手。请为以下内容推荐景点：
    上下文： {context}
    Query: {query}
    推荐有趣的游览地点，包含描述、亮点和实用信息。"""

    response = client.models.generate_content(
        model=FLASH_MODEL,
        contents=prompt,
    )
    return response.text.strip()

In [ ]:
@tracer.chain()
def determine_next_step(user_query, context, cycle, max_cycles):
    """
    根据当前上下文与用户查询确定下一步调用的智能体。
    Args:
        user_query: 初始用户查询
        context: 当前累积上下文
        cycle: 当前循环编号
        max_cycles: 最大智能体调用次数
    Returns:
        接下来要调用的函数名
    """
    orchestration_prompt = f"""你是编排智能体。请决定下一步：
    用户查询： {user_query}
    当前上下文： {context}
    当前循环： {cycle}/{max_cycles}
    选择一个可用工具帮助处理用户查询，或决定返回最终答案。
    """

    # 定义编排器工具
    orchestrator_tools = {
        "function_declarations": [
            {
                "name": "call_planning_agent",
                "description": "调用规划智能体，创建包含后续步骤的结构化计划",
            },
            {
                "name": "call_flight_planning_agent",
                "description": "调用航班规划智能体，帮助查找与推荐航班",
            },
            {
                "name": "call_hotel_recommendation_agent",
                "description": "调用酒店推荐智能体，推荐住宿",
            },
            {
                "name": "call_travel_attraction_agent",
                "description": "调用旅游景点智能体，推荐有趣的游览地点并提供描述",
            },
            {
                "name": "call_user_proxy_agent",
                "description": "调用扮演用户并提供反馈的用户代理智能体",
            },
            {
                "name": "return_final_answer",
                "description": "收集到足够信息后，向用户返回最终答案",
            },
        ]
    }

    orchestration_response = client.models.generate_content(
        model=FLASH_MODEL,
        contents=orchestration_prompt,
        config=types.GenerateContentConfig(tools=[orchestrator_tools]),
    )

    if orchestration_response.candidates[0].content.parts[0].function_call:
        function_call = orchestration_response.candidates[0].content.parts[0].function_call
        return function_call.name
    else:
        return "return_final_answer"  # 没有工具调用时默认返回最终答案


@tracer.chain()
def execute_agent_call(function_name, user_query, context):
    """
    执行指定智能体调用，并返回回复与智能体类型。
    Args:
        function_name: 要调用的函数名
        user_query: 初始用户查询
        context: 当前累积上下文
    Returns:
        (agent_response, agent_type) 元组
    """
    if function_name == "call_flight_planning_agent":
        agent_response = call_flight_planning_agent(user_query, context)
        agent_type = "航班规划"
    elif function_name == "call_hotel_recommendation_agent":
        agent_response = call_hotel_recommendation_agent(user_query, context)
        agent_type = "酒店推荐"
    elif function_name == "call_travel_attraction_agent":
        agent_response = call_travel_attraction_agent(user_query, context)
        agent_type = "旅游景点"
    elif function_name == "call_user_proxy_agent":
        agent_response = call_user_proxy_agent(user_query, context)
        agent_type = "用户代理"
    else:
        agent_response = ""
        agent_type = "Unknown"

    return agent_response, agent_type


@tracer.chain()
def generate_final_answer(user_query, context, max_cycles_reached=False):
    """
    根据累积上下文生成最终答案。
    Args:
        user_query: 初始用户查询
        context: 当前累积上下文
        max_cycles_reached: 是否达到最大循环次数
    Returns:
        给用户的最终回复
    """
    final_prompt = f"""为用户查询生成最终回复： {user_query}
    根据以下上下文： {context}
    """

    if max_cycles_reached:
        final_prompt += "\n\n提供全面且有帮助的答案，并说明已达到最大处理循环次数。"
    else:
        final_prompt += "\n\n提供全面且有帮助的答案。"

    final_response = client.models.generate_content(
        model=FLASH_MODEL,
        contents=final_prompt,
    )

    return final_response.text.strip()


@tracer.agent()
def orchestrator(user_query, max_cycles=3):
    """
    编排器，决定流程各步骤应调用哪个智能体。
    Args:
        user_query: 初始用户查询
        max_cycles: 返回用户前允许的最大智能体调用次数
    Returns:
        给用户的最终回复
    """
    context = ""
    cycle = 0

    while cycle < max_cycles:
        # 确定下一步
        function_name = determine_next_step(user_query, context, cycle, max_cycles)

        if function_name == "return_final_answer":
            return generate_final_answer(user_query, context)

        # 执行智能体调用
        agent_response, agent_type = execute_agent_call(function_name, user_query, context)

        # 使用智能体回复更新上下文
        context += f"\n\n{agent_type} 智能体输出：\n{agent_response}"
        cycle += 1

    # 达到最大循环次数时返回当前结果
    return generate_final_answer(user_query, context, max_cycles_reached=True)


# 使用示例
user_query = """I want to plan a 5-day trip to Paris, France, sometime in October. I'm interested
in museums and good food. Find flight options from SFO, suggest mid-range hotels near the city center,
and recommend some relevant activities."""

response = orchestrator(user_query)
print(response)